In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
import math
import cmath

# parameter definitions

In [10]:
def make_omega_array(N):
    omega_array = np.random.normal(0, 1, N)
    return omega_array

In [11]:
def make_theta_array(N):
    theta_array = np.random.uniform(-np.pi, np.pi, N)
    return theta_array

In [28]:
N=500 #Number of oscillator
K_array=[0,3,5] #Coupling strength
F_array=[0,3,5] #Forcing strengths
number_time_steps=500 #Number of time steps
time_step=0.01 
Omega_array=[0,3,5] #Natural frequency of sun in hours
omega_array=make_omega_array(N)#makes array of oscillators natural frequencies
theta_array=make_theta_array(N)#makes array of oscillator phases

# functions

In [13]:
#the rhs calculation
def forced_kuramoto(theta_array, omega_array, K, N, F, Omega):
    theta_dot = np.zeros(N)
    sum_cisj=0
    for j in range(N):
        sum_cisj+=math.cos(theta_array[j])+1j*math.sin(theta_array[j])
    z=(1/N)*sum_cisj
    for i in range(N):
        complex_sum_replacement=K*z*(math.cos(theta_array[i])-1j*math.sin(theta_array[i]))
        img_sum_replacement=complex_sum_replacement.imag
        theta_dot[i] = (omega_array[i]-Omega) + img_sum_replacement - F * math.sin(theta_array[i])

    return theta_dot

In [14]:
def runge_kutta_4th_order(theta_array, omega_array, K, N, F, Omega, delta_t):
    k1= forced_kuramoto(theta_array, omega_array, K, N, F, Omega)*delta_t
    k2= forced_kuramoto(theta_array + 0.5*k1, omega_array, K, N, F, Omega)*delta_t
    k3= forced_kuramoto(theta_array + 0.5*k2, omega_array, K, N, F, Omega)*delta_t
    k4= forced_kuramoto(theta_array + k3, omega_array, K, N, F, Omega,)*delta_t
    
    theta_array_plustimestep = theta_array + (1/6)*(k1 + 2*k2 + 2*k3 + k4)
    
    return theta_array_plustimestep

In [15]:
def runge_kutta_loop(theta_array, omega_array, K, N, F, Omega, delta_t, total_T):
    
    theta_history = np.zeros((total_T, len(theta_array)))
    theta_history[0] = theta_array
    
    for i in range(1, total_T):
        theta_history[i] = runge_kutta_4th_order(theta_history[i-1], omega_array, K, N, F, Omega, delta_t)

    return theta_history

In [17]:
def calculate_order_parameter_rhs(N, theta_history, timestep):
    # 1/N sum from l=1 to N cos(theta_l) + isin(theta_l)
    theta_l_sum = 0
    for i in range(N):
        theta_l = theta_history[timestep, i]
        theta_l_sum += math.cos(theta_l)+ 1j*math.sin(theta_l)
    order_parameter_rhs = (1/N)*theta_l_sum
    return order_parameter_rhs

In [18]:
def calculate_order_parameter_lhs(order_parameter_rhs): 
    #R(t) e^(i PSi(t)=order_paramter_rhs=Z(t)
    #R is magnitude of Z(t)
    #Psi(t) is the argument of Z(t)        Re(z)=|z|cos(arg)  Re(z)/|z|=cos(arg)    cos^-1( Re(z)/|z|)=arg?
    #e^iPsi(t)= cos(Psi(t))+ i sin(Psi(t))
    re = order_parameter_rhs.real
    im = order_parameter_rhs.imag
    R = math.sqrt(re**2 + im**2)
    Psi = cmath.phase(order_parameter_rhs)
    return R, Psi

In [6]:
# this function is new, it does not d everything yet
def classify_region(r_array, psi_array, tt):
    #tt=transient time
    psi_array_after_tt=psi_array[tt:]
    r_array_after_tt=r_array[tt:]
    
    #less decimal places please
    psi_array_after_tt=np.round(psi_array_after_tt,3)
    r_array_after_tt=np.round(r_array_after_tt,3)
    
    #classify R
    r_monotone = True
    for r in range(len(r_array_after_tt)-1):
        if r_array_after_tt[r+1] < r_array_after_tt[r]: #if at any point, r+1 is less that r, then r is not monotone
            r_monotone = False
            break
    r_mean = np.mean(r_array_after_tt) #this is just for printing info for now - seeing how similar r at the last timestep it to the mean r throughout
    print(f"mean order paramter = {r_mean}")
    print(f"order parameter at end = {r_array_after_tt[299]}") #OBS!!! the number here is the last index of the array after tt cut. You need to change this
                                                            #if you change lengths of original arrays, or tt. This should be improved so it is not hard-coded
                                                            #in, but a flexible part of the function
            
    #classify psi
    tolerated_diff=1e-6 #psi is unlikely to be identical to all decimal places at all times. I've therefore said it only has to be identical to some small decimal
    diffs = np.diff(psi_array_after_tt) #array of differences between each psi
    psi_constant = np.all(np.abs(diffs) < tolerated_diff) #psi non constant if these differences are more than something very tiny
    
    #region classification
    if r_monotone: 
        print("region A, or region D if too much transient time cut")
    elif (not psi_constant):
        print("region E or E*")
    else: 
        print("r oscillating, but psi constant - some unknown region")
    
    print() #prints an empty line to seperate runs

# testing

In [4]:
#I have not made the main loop into a function yet but it does loop over 3D grid
for i in range(len(F_array)):
    for j in range(len(Omega_array)):
        for k in range(len(K_array)):
            K=K_array[i]
            Omega=Omega_array[j]
            F=F_array[k]
            theta_history=runge_kutta_loop(theta_array, omega_array, K, N, F, Omega, time_step, number_time_steps)
                                  
            r_array=[]
            psi_array=[]
                                  
            for t in range(number_time_steps):
                order_parameter_rhs = calculate_order_parameter_rhs(N, theta_history, t)
                R, Psi = calculate_order_parameter_lhs(order_parameter_rhs)
                r_array.append(R)
                psi_array.append(Psi)
            psi_array = np.unwrap(psi_array) #allows it to increase monotonically rather than wrap around at 2pi
            print(f"One loop complete, K={K}, Omega={Omega}, F={F}")
            classify_region(r_array, psi_array, 200) #tt is set to something random to make testing shorter
            #print(r_array)
            #print(psi_array)

NameError: name 'F_array' is not defined